[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/35_bpe.ipynb)

# 🔴 Hard: Byte-Pair Encoding (BPE)

Implement a simple **BPE tokenizer** — the foundation of GPT/LLaMA tokenization.

### Signature
```python
class SimpleBPE:
    def __init__(self): ...
    def train(self, corpus: list[str], num_merges: int): ...
    def encode(self, text: str) -> list[str]: ...
```

### Algorithm (training)
1. Split each word into characters + `</w>` end marker
2. Count all adjacent pairs across the corpus
3. Merge the most frequent pair into a single token
4. Repeat for `num_merges` iterations

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [37]:
class SimpleBPE:
    def __init__(self):
        self.merges = []
        self.single_chars = {}

    def _merge(self, text, pair):
      i = 0
      n = len(text)
      new_tokens = []
      #scan left-to-right and merge most_frequent pair
      while i < n - 1:
        curr, next = text[i], text[i + 1]
        if (curr, next) == pair:
          new_tokens.append(curr + next)
          i += 2
        else:
          new_tokens.append(curr)
          i += 1

      if i == n - 1:
        new_tokens.append(text[i])

      return new_tokens

    def train(self, corpus, num_merges):
        tokens = []#start with chars as toks
        for word in corpus:
          tokens.extend([c for c in word])
          tokens.append("</w>")

        self.single_chars = set(tokens)

        if len(tokens) < 2:
          return

        #count token pairs
        for _ in range(num_merges):

          pairs = [(a, b) for a, b in zip(tokens[:-1], tokens[1:]) if not a.endswith("</w>")] #avoid merging across words
          if not pairs: #when only one token there is no pair
            return

          count = {}
          for pair in pairs:
            count[pair] = count.get(pair, 0) + 1

          most_freq = min(
              pairs,
              key= lambda x: (-count[x], x)
          ) #get most_frequent (minimum -count) with lexicographic tiebreaker (minimum lexic. pair)

          self.merges.append(most_freq)

          tokens = self._merge(tokens, most_freq)



    def encode(self, text):
        #in the classic BPE we have this training word-based (/w token) with a merge priority based (order of merges in self.merges) and not greedy right-to-left based
        #so we only need self.merges (and self.single_chars) to create tokens, then a vocabulary is needed to link each token with its ID (to then obtain the embedding)
        unknown = set([c for word in text.split() for c in word]) - self.single_chars

        if unknown:
          raise ValueError("Unknown symbol in text")

        #at the end I have a list of merges, which order is the priority of the merges on the text (last priority is single char, in fact I have single_chars set just to check if all symbols in each word is present or not in the voc.)
        code = []

        #iterate over words to avoid merges across different words
        for word in text.split():
          tokens = list(word) + ["</w>"]

          for pair in self.merges:
            tokens = self._merge(tokens, pair)

          code.extend(tokens) #append single tokens instead of list of tokens

        return code





In [38]:
# 🧪 Debug
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges[:5])
print('Encode:', bpe.encode('low lower'))

Merges: [('l', 'o'), ('lo', 'w'), ('low', '</w>'), ('e', 's'), ('es', 't')]
Encode: ['low</w>', 'low', 'er</w>']


In [39]:
# ✅ SUBMIT
from torch_judge import check
check('bpe')


🧪 Testing: Byte-Pair Encoding (BPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Correct number of merges (0.3ms)
  ✅ [2/4] Most frequent pair merged first (0.1ms)
  ✅ [3/4] Encode returns list of strings (0.5ms)
  ✅ [4/4] More merges -> fewer tokens (0.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (1.3ms total)
  Progress saved. Run status() to see your dashboard.

